# 날씨의 ED아이 — 기상 데이터 정규화 기반 KTCI & 여행 쾌적도 (기상팀 통합 코드)

**DSL 26-2 기상팀** · 백가은(팀장)·최유성·이은민·신동준·김현희

이 노트북은 프로젝트 핵심 분석 파이프라인을 한 흐름으로 묶은 것입니다.

1. **정규화** : ASOS·AWS·AirKorea 원자료 → 인간활동 스트레스 지수(0~1) `01_normalize_weather.py`
2. **마스터 결합** : 관측소→시군구 매핑 후 날씨 × 관광 마스터 생성 `02_build_master.py`
3. **히든스팟** : `히든점수 = 매력도 − 혼잡도` 로 계절별 히든스팟 도출 `03_hidden_spot.py`
4. **데이터 기반 KTCI** : 계절별 가중치 산출·검증 `KTCI_code/src/run_pipeline.py` (최유성)
5. **오늘의 날씨 점수** : 실시간 시군구 점수·5등급·기상위험 하향 `service/app/weather_today.py` (이은민)
6. **여행 일정 동선 점수** : 예보 기반 일자별 동선 채점 `service/app/trip.py` (신동준)

> ⚠️ 원본 관측 자료는 학교 서버(`/data/InSitu/…`)에 **읽기 전용**으로 존재합니다.
> 각 셀의 입력 경로는 실제 실행 환경 기준이며, 제출 저장소에는 산출물 샘플(`Dataset/`)만 포함됩니다.
> 데이터 기반 KTCI 가중치 산출·검증은 `KTCI_code/` 를 참고하세요.

## 0. 정규화 앵커 설정 (`config.py`)

5개 영역(기온·습도·바람·강수·대기질)의 '적합·주의·해로움·위험' 경계값. 국내외 공인 지수·특보 기준.

In [ ]:
# -*- coding: utf-8 -*-
"""
config.py — 정규화 앵커(임계값) 정의
====================================
5개 영역의 '적합 · 주의 · 해로움 · 위험' 경계값. 모두 국내외 공인 지수·특보 기준에서 가져옴.
방향 : 0 = 좋음(쾌적),  1 = 나쁨(위험).  구간선형 보간의 x(관측값)–s(스트레스) 앵커.
"""

# (x 관측값, s 스트레스) 앵커 쌍 — 구간선형 보간용
ANCHORS = {
    # 기온: 체감온도 / UTCI 인체 열수지 기준 (여름 더위 방향; 겨울은 풍속냉각 WCT로 별도 산출)
    "thermal_heat": {
        "unit": "체감온도(℃)",
        "ref": "UTCI · 기상청 체감온도",
        "points": [(9, 0.0), (26, 0.33), (33, 0.66), (38, 0.90), (45, 1.0)],
        "labels": ["적합", "주의", "해로움", "위험"],
    },
    # 습도: 불쾌지수 THI (Thom, 1959).  THI = 1.8T − 0.55(1−RH/100)(1.8T−26) + 32
    "humidity": {
        "unit": "THI 지수",
        "ref": "불쾌지수 THI (Thom, 1959)",
        "points": [(68, 0.0), (75, 0.33), (80, 0.66), (83, 1.0)],
        "labels": ["쾌적", "주의", "불쾌", "매우불쾌"],
        "comfort_RH": (40, 60),
    },
    # 바람: 보퍼트 풍력계급 · 기상청 강풍특보 (주의보 14㎧, 경보 21㎧)
    "wind": {
        "unit": "풍속(㎧)",
        "ref": "보퍼트 · 기상청 강풍특보",
        "points": [(0, 0.0), (8, 0.33), (14, 0.66), (21, 0.90), (30, 1.0)],
        "labels": ["적합", "주의", "해로움", "위험"],
    },
    # 강수: 기상청 호우특보 · 강수강도 분류 (호우주의보 80㎜/일)
    "precipitation": {
        "unit": "일강수량(㎜)",
        "ref": "기상청 호우특보",
        "points": [(0, 0.0), (10, 0.33), (50, 0.66), (80, 0.90), (150, 1.0)],
        "labels": ["적합", "주의", "해로움", "위험"],
    },
}

# 대기질: 통합대기환경지수(CAI) — 물질별 구간지수 후 '최악 물질' 대표
#   PM2.5 / PMc(=PM10−PM2.5) 분리. 경계는 CAI·WHO·EPA 기준.
AIR_BREAKPOINTS = {
    "PM25": [15, 35, 75],     # ㎍/㎥  좋음|보통|나쁨|매우나쁨
    "PM10": [30, 80, 150],
    "O3":   [0.030, 0.090, 0.150],   # ppm
    "NO2":  [0.030, 0.060, 0.200],
    "SO2":  [0.020, 0.050, 0.150],
    "CO":   [2, 9, 15],       # ppm
}

# 결합 규칙
COMBINE = {
    "thermal_humidity": "worse-max",     # 더위·추위(건조·과습) 중 나쁜 쪽
    "air_quality": "worst-pollutant",    # 6종 중 최악 물질 대표 (CAI 방식)
    "missing": "renormalize (관측 2개 미만이면 미산출)",
}


## 1. 기상 스트레스 정규화 (`01_normalize_weather.py`)

시간별 관측 → 일 단위 집계 → 결측 처리 → 체감온도·THI·PMc 파생 → **5영역 구간선형 정규화**(0=좋음/1=나쁨).

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
normalize_weather.py
====================
전국 일별 기상데이터(ASOS / AWS / AirKorea) 정규화 파이프라인.
방법론 문서 v0.1 구현 — 5개 영역(기온·습도·대기질·바람·강수)을 0-1로 정규화.
방향: 0 = 좋음/쾌적,  1 = 나쁨/위험 (위해·스트레스 지수)

처리 순서
    소스별 원본(시간자료) 로드 → 결측 마스킹 → 일별 집계 → 파생(체감온도·THI·PMc)
    → 선행연구 앵커 기반 구간선형 정규화 → 소스별 산출 CSV 저장

★ 안전 ★
    - 서버 원본(/data/InSitu/…)은 읽기 전용으로만 접근한다.
    - 산출물은 --output 폴더에만 쓰며, 소스 폴더 내부이면 실행을 거부한다.

의존성 : pandas, numpy  (필수)
    pip install pandas numpy

사용법
    python normalize_weather.py --source all
    python normalize_weather.py --source asos --output ./out
    python normalize_weather.py --source airkorea --limit-files 2   # 테스트용
"""
from __future__ import annotations
import argparse, os, sys, glob, warnings
import numpy as np
import pandas as pd

# 전부 결측인 날의 nanmax/nanmean 경고 억제 (해당 날은 정상적으로 NaN 유지)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ===========================================================================
# 0. 설정 : 경로 · 결측코드 · 유효자료 기준
# ===========================================================================
ROOTS = {
    "asos":     "/data/InSitu/KMA_ASOS/raw",
    "aws":      "/data/InSitu/KMA_AWS/month",
    "airkorea": "/data/InSitu/AirKorea/raw_data",
}
NA_VALUES = {
    "asos":     [-9, -9.0, -99, -99.0, "-9", "-9.0", "-99.0", "-99.00", "-9.00"],
    "aws":      [-99, -99.0, "-99", "-99.0"],
    "airkorea": [-999, -999.0, "-999", ""],
}
MIN_VALID_HOURS = 18   # 하루 24시간 중 유효 관측 최소치 (75%)

# ===========================================================================
# 1. 정규화 앵커 (방법론 문서 v0.1) — (xs 농도/값, ys 스트레스 0-1)
# ===========================================================================
A_TEMP_HEAT = ([26, 32, 35, 38, 46], [0.0, 0.40, 0.55, 0.70, 1.0])          # 체감온도(℃)→스트레스
A_TEMP_COLD = ([-40, -27, -13, 0, 9], [1.0, 0.80, 0.55, 0.30, 0.0])         # 체감온도(℃)→스트레스
A_RH_DRY    = ([0, 10, 20, 40], [1.0, 1.0, 0.6, 0.0])                        # RH%(<40)
A_RH_HUMID  = ([60, 80, 95, 100], [0.0, 0.5, 0.9, 1.0])                      # RH%(>60)
A_THI       = ([68, 75, 80, 83], [0.0, 0.5, 0.8, 1.0])                       # 불쾌지수
A_PM25      = ([0, 15, 35, 75, 150], [0, .25, .5, .75, 1.0])                 # ㎍/㎥
A_PMC       = ([0, 15, 45, 75, 150], [0, .25, .5, .75, 1.0])                 # ㎍/㎥ (조대분획)
A_PM10      = ([0, 30, 80, 150, 300], [0, .25, .5, .75, 1.0])               # ㎍/㎥ (PM2.5 미측정기 fallback)
A_O3        = ([0, .03, .09, .15, .30], [0, .25, .5, .75, 1.0])              # ppm (8h)
A_NO2       = ([0, .03, .06, .20, .40], [0, .25, .5, .75, 1.0])              # ppm
A_SO2       = ([0, .02, .05, .15, .30], [0, .25, .5, .75, 1.0])              # ppm
A_CO        = ([0, 2, 9, 15, 30], [0, .25, .5, .75, 1.0])                    # ppm
A_WIND      = ([3.3, 8.0, 13.9, 17.2, 21, 24.5], [0, .25, .5, .70, .85, 1.0])  # m/s
A_RN_AMT    = ([0, 10, 30, 50, 80, 150], [0, .2, .4, .6, .8, 1.0])           # ㎜/일
A_RN_INT    = ([0, 3, 15, 30, 50], [0, .25, .5, .75, 1.0])                   # ㎜/h


def interp_clip(x, anchors):
    """구간선형 보간 후 [0,1] 클리핑. NaN 은 NaN 유지."""
    xs, ys = anchors
    return np.clip(np.interp(x, xs, ys), 0.0, 1.0)


# ===========================================================================
# 2. 파생 지표 : 체감온도 · 불쾌지수
# ===========================================================================
def wind_chill(T, WS_ms):
    """겨울 풍속냉각 체감온도(기상청/Environment Canada). T<=10℃ & 바람 유효시 적용."""
    T = np.asarray(T, float); WS_ms = np.asarray(WS_ms, float)
    V = np.maximum(WS_ms, 0) * 3.6  # km/h
    wct = 13.12 + 0.6215 * T - 11.37 * np.power(np.maximum(V, 0.1), 0.16) \
        + 0.3965 * T * np.power(np.maximum(V, 0.1), 0.16)
    use = (T <= 10) & (V >= 4.8)
    return np.where(use, wct, T)


def heat_index(T, RH):
    """여름 열지수 체감온도(NWS Rothfusz). T>=27℃ 에서 적용, 그 외 기온 그대로."""
    T = np.asarray(T, float); RH = np.asarray(RH, float)
    Tf = T * 9.0 / 5.0 + 32.0
    HI = (-42.379 + 2.04901523 * Tf + 10.14333127 * RH
          - 0.22475541 * Tf * RH - 6.83783e-3 * Tf**2 - 5.481717e-2 * RH**2
          + 1.22874e-3 * Tf**2 * RH + 8.5282e-4 * Tf * RH**2 - 1.99e-6 * Tf**2 * RH**2)
    HI_c = (HI - 32.0) * 5.0 / 9.0
    return np.where(T >= 27, np.maximum(HI_c, T), T)


def discomfort_index(T, RH):
    """불쾌지수 THI (기상청식). T ℃, RH %."""
    T = np.asarray(T, float); RH = np.asarray(RH, float)
    return 1.8 * T - 0.55 * (1 - RH / 100.0) * (1.8 * T - 26.0) + 32.0


def rh_deviation(rh):
    """상대습도 쾌적대(40-60%) 이탈 스트레스."""
    rh = np.asarray(rh, float)
    dry = interp_clip(rh, A_RH_DRY)
    humid = interp_clip(rh, A_RH_HUMID)
    dev = np.where(rh < 40, dry, np.where(rh > 60, humid, 0.0))
    return np.where(np.isnan(rh), np.nan, dev)


# ===========================================================================
# 3. 기상 소스(ASOS/AWS) : 일별 집계 + 정규화
# ===========================================================================
def _daily_met(df, colmap):
    """시간자료 → (station,date) 일별 집계."""
    dt = df[colmap["datetime"]].astype("int64").astype(str)
    df = df.assign(_date=dt.str[:8], _stn=df[colmap["station"]])
    g = df.groupby(["_stn", "_date"])
    out = pd.DataFrame({
        "TA_mean": g[colmap["TA"]].mean(), "TA_max": g[colmap["TA"]].max(),
        "TA_min":  g[colmap["TA"]].min(),  "n_TA": g[colmap["TA"]].count(),
        "HM_mean": g[colmap["HM"]].mean(), "n_HM": g[colmap["HM"]].count(),
        "WS_mean": g[colmap["WS"]].mean(), "WS_max": g[colmap["WS"]].max(),
        "n_WS": g[colmap["WS"]].count(),
        "RN_sum": g[colmap["precip"]].sum(min_count=1),
        "RN_max": g[colmap["precip"]].max(), "n_RN": g[colmap["precip"]].count(),
    }).reset_index().rename(columns={"_stn": "station", "_date": "date"})
    return out


def _mask_invalid(daily):
    """유효 관측시간 미만 변수는 NaN 처리."""
    m = MIN_VALID_HOURS
    for var, ncol in [("TA_mean", "n_TA"), ("TA_max", "n_TA"), ("TA_min", "n_TA"),
                      ("HM_mean", "n_HM"), ("WS_mean", "n_WS"), ("WS_max", "n_WS")]:
        daily.loc[daily[ncol] < m, var] = np.nan
    # 강수는 결측을 0으로 오인하면 안 되므로 유효시간 부족시 NaN
    daily.loc[daily["n_RN"] < m, ["RN_sum", "RN_max"]] = np.nan
    return daily


def normalize_met(daily):
    """기상 4개 영역 정규화."""
    # 파생
    AT_day = heat_index(daily["TA_max"], daily["HM_mean"])
    AT_night = wind_chill(daily["TA_min"], daily["WS_mean"])
    THI = discomfort_index(daily["TA_mean"], daily["HM_mean"])
    daily["AT_day"] = AT_day
    daily["AT_night"] = AT_night
    daily["THI"] = THI
    # ① 기온 : 열/냉 스트레스의 max
    s_heat = interp_clip(AT_day, A_TEMP_HEAT)
    s_cold = interp_clip(AT_night, A_TEMP_COLD)
    daily["S_temp"] = np.nanmax(np.vstack([s_heat, s_cold]), axis=0)
    # ② 습도 : RH 이탈 / THI 의 max
    s_rh = rh_deviation(daily["HM_mean"])
    s_thi = interp_clip(THI, A_THI)
    daily["S_humidity"] = np.nanmax(np.vstack([s_rh, s_thi]), axis=0)
    # ④ 바람 : 일최대풍속 기준
    daily["S_wind"] = interp_clip(daily["WS_max"], A_WIND)
    # ⑤ 강수 : 강수량 / 강도 의 max (무강수일=0)
    s_amt = interp_clip(daily["RN_sum"], A_RN_AMT)
    s_int = interp_clip(daily["RN_max"], A_RN_INT)
    daily["S_precip"] = np.nanmax(np.vstack([s_amt, s_int]), axis=0)
    return daily


def process_met(source, files, out_path):
    colmaps = {
        "asos": dict(datetime="KST", station="STN", TA="TA", HM="HM", WS="WS", precip="RN"),
        "aws":  dict(datetime="KST", station="STN", TA="TA", HM="HM", WS="WS", precip="RN_HR1"),
    }
    cm = colmaps[source]
    need = list(dict.fromkeys(cm.values()))
    dailies = []
    for i, fp in enumerate(files, 1):
        try:
            df = pd.read_csv(fp, encoding="utf-8-sig", na_values=NA_VALUES[source])
            df.columns = [c.strip() for c in df.columns]
            miss = [c for c in need if c not in df.columns]
            if miss:
                print(f"    (건너뜀) {os.path.basename(fp)} : 컬럼없음 {miss}"); continue
            dailies.append(_daily_met(df[need], cm))
            print(f"    [{i}/{len(files)}] {os.path.basename(fp)}  일수집계 {len(dailies[-1])}")
        except Exception as e:
            print(f"    (오류 건너뜀) {os.path.basename(fp)} : {e}")
    if not dailies:
        print("    처리할 데이터 없음"); return
    daily = pd.concat(dailies, ignore_index=True)
    # 월경계 없이 파일=월 단위라 (station,date) 중복 없음. 안전차원 재집계 생략.
    daily = _mask_invalid(daily)
    daily = normalize_met(daily)
    cols = ["station", "date", "S_temp", "S_humidity", "S_wind", "S_precip",
            "TA_mean", "TA_max", "TA_min", "HM_mean", "WS_mean", "WS_max",
            "RN_sum", "RN_max", "AT_day", "AT_night", "THI",
            "n_TA", "n_HM", "n_WS", "n_RN"]
    daily = daily.sort_values(["station", "date"])[cols].round(4)
    daily.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"    ▶ 저장: {out_path}  ({len(daily):,}행, 지점 {daily['station'].nunique()}개)")


# ===========================================================================
# 4. AirKorea : 일별 집계 + 대기질 정규화
# ===========================================================================
AK_COLS7 = ["station", "dt", "SO2", "CO", "O3", "NO2", "PM10"]
AK_COLS8 = ["station", "dt", "SO2", "CO", "O3", "NO2", "PM25", "PM10"]


def _daily_airkorea(df):
    """시간자료 → (station,date) 일별. O3 는 일 최대 8시간 이동평균, 나머지 일평균."""
    df = df.copy()
    df["dt"] = df["dt"].astype("int64").astype(str)
    df["date"] = df["dt"].str[:8]
    df["hour"] = df["dt"].str[8:10]
    gases = [c for c in ["SO2", "CO", "NO2", "PM25", "PM10"] if c in df.columns]
    g = df.groupby(["station", "date"])
    agg = {c: "mean" for c in gases}
    daily = g.agg(agg)
    daily["n_obs"] = g["dt"].count()
    # O3 8시간 이동평균의 일 최대
    df = df.sort_values(["station", "dt"])
    o3_8h = df.groupby("station")["O3"].transform(lambda s: s.rolling(8, min_periods=6).mean())
    df["_O3_8h"] = o3_8h
    o3d = df.groupby(["station", "date"])["_O3_8h"].max()
    daily["O3"] = o3d
    return daily.reset_index()


def normalize_airkorea(daily):
    # 입자상 처리 : PM2.5 있는 행은 PM2.5+PMc, 없는 행(2001-2018)은 PM10 fallback.
    # 행 단위로 분기해 초기연도 미세먼지 누락을 방지한다.
    subs = {}
    if "PM25" in daily.columns:
        pm25 = daily["PM25"]
        pmc = np.maximum(daily["PM10"] - pm25, 0.0)
        daily["PMc"] = pmc
        pm25_missing = pm25.isna().values
        s_pm10_fb = interp_clip(daily["PM10"], A_PM10)
        subs["pm25"] = interp_clip(pm25, A_PM25)
        subs["pmc"]  = interp_clip(pmc, A_PMC)
        # PM2.5 결측 행에서만 PM10 fallback 사용(측정된 행은 NaN 처리해 이중계산 방지)
        subs["pm10"] = np.where(pm25_missing, s_pm10_fb, np.nan)
    else:
        subs["pm10"] = interp_clip(daily["PM10"], A_PM10)
    subs.update({
        "o3":  interp_clip(daily["O3"], A_O3),
        "no2": interp_clip(daily["NO2"], A_NO2),
        "so2": interp_clip(daily["SO2"], A_SO2),
        "co":  interp_clip(daily["CO"], A_CO),
    })
    S = pd.DataFrame(subs, index=daily.index)
    for k in S.columns:
        daily[f"s_{k}"] = S[k].round(4)
    daily["S_airquality"] = np.nanmax(S.values, axis=1).round(4)      # 최악물질
    daily["S_aq_mean"] = np.nanmean(S.values, axis=1).round(4)         # 참고: 평균
    return daily


def process_airkorea(files, out_path):
    dailies = []
    for i, fp in enumerate(files, 1):
        try:
            first = pd.read_csv(fp, header=None, nrows=1)
            ncol = first.shape[1]
            names = AK_COLS8 if ncol == 8 else AK_COLS7
            df = pd.read_csv(fp, header=None, names=names,
                             na_values=NA_VALUES["airkorea"])
            dailies.append(_daily_airkorea(df))
            print(f"    [{i}/{len(files)}] {os.path.basename(fp)} ({ncol}열)  일수집계 {len(dailies[-1])}")
        except Exception as e:
            print(f"    (오류 건너뜀) {os.path.basename(fp)} : {e}")
    if not dailies:
        print("    처리할 데이터 없음"); return
    daily = pd.concat(dailies, ignore_index=True)
    # 분기/월 파일이라 같은 (station,date) 중복 없음 → 재집계 불필요
    daily.loc[daily["n_obs"] < MIN_VALID_HOURS,
              [c for c in ["SO2", "CO", "O3", "NO2", "PM25", "PM10"] if c in daily.columns]] = np.nan
    daily = normalize_airkorea(daily)
    base = ["station", "date", "S_airquality", "S_aq_mean"]
    scols = [c for c in daily.columns if c.startswith("s_")]
    raw = [c for c in ["PM25", "PMc", "PM10", "O3", "NO2", "SO2", "CO", "n_obs"] if c in daily.columns]
    daily = daily.sort_values(["station", "date"])[base + scols + raw].round(4)
    daily.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"    ▶ 저장: {out_path}  ({len(daily):,}행, 측정소 {daily['station'].nunique()}개)")


# ===========================================================================
# 5. main
# ===========================================================================
def list_files(source):
    root = ROOTS[source]
    if source == "airkorea":
        return sorted(glob.glob(os.path.join(root, "*", "*.csv")))
    return sorted(glob.glob(os.path.join(root, "*.csv")))


def is_subpath(child, parent):
    try:
        return os.path.commonpath([os.path.realpath(child), os.path.realpath(parent)]) == os.path.realpath(parent)
    except Exception:
        return False


def main():
    ap = argparse.ArgumentParser(description="기상데이터 5영역 정규화 (0=좋음,1=나쁨)")
    ap.add_argument("--source", choices=["asos", "aws", "airkorea", "all"], default="all")
    ap.add_argument("--output", default="./normalized_output")
    ap.add_argument("--limit-files", type=int, default=0, help="소스별 처리 파일 수 제한(테스트)")
    args = ap.parse_args()

    out_abs = os.path.realpath(args.output)
    for r in ROOTS.values():
        if os.path.exists(r) and is_subpath(out_abs, r):
            print(f"[중단] 출력 폴더가 소스 내부입니다: {out_abs}"); sys.exit(1)
    os.makedirs(out_abs, exist_ok=True)

    sources = ["asos", "aws", "airkorea"] if args.source == "all" else [args.source]
    for src in sources:
        print(f"\n=== {src.upper()} ===")
        files = list_files(src)
        if args.limit_files:
            files = files[:args.limit_files]
        if not files:
            print(f"    파일 없음: {ROOTS[src]}"); continue
        out_path = os.path.join(out_abs, f"{src}_normalized.csv")
        if src == "airkorea":
            process_airkorea(files, out_path)
        else:
            process_met(src, files, out_path)
    print("\n완료.")


if __name__ == "__main__":
    main()


## 2. 날씨 × 관광 마스터 결합 (`02_build_master.py`)

정규화 결과 + 관광 지표를 `(signguCode, baseYm)` 키로 결합. 관측소→시군구 매핑(264개 시군구 100%) 포함.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
build_master.py — 관광데이터 × 정규화 기상·대기질 마스터 CSV 생성
키: (signguCode, date). 관광(현지인/외지인/외국인/합계) + 기상 S_* + 대기질 S_*.
- 기상: ASOS+AWS를 시군구 기초단위(광역시는 시도풀)로 배정, 지점 평균.
- 대기질: AirKorea 측정소를 동일 규칙으로 배정, 평균.
- 광역시 자치구는 해당 시(도) 평균을 공유(도시규모 기상/대기질).
"""
import pandas as pd, numpy as np, re, sys

TOUR="/mnt/user-data/uploads/Desktop/tour_visitor_locgo_sigungu_daily_2023_2025_merged.csv"
META="/root/.claude/uploads/c1b7af1e-dfdd-5d68-832d-7596ebedd996/e2e066a9-META________20260716163938.csv"
AK_KEYSRC="/tmp/akpeek/2026#Ub144 1#Uc6d4.xlsx"
BASE="/home/claude/normalized_2023_2026/"

SIDO_PREFIX=[('서울','11'),('부산','26'),('대구','27'),('인천','28'),('광주광역','29'),('대전','30'),
 ('울산','31'),('세종','36'),('경기','41'),('충청북','43'),('충북','43'),('충청남','44'),('충남','44'),
 ('전라남','46'),('전남','46'),('경상북','47'),('경북','47'),('경상남','48'),('경남','48'),
 ('제주','50'),('강원','51'),('전라북','52'),('전북','52')]
METRO={'11','26','27','28','29','30','31','36'}

def sido_of(prov):
    for k,v in SIDO_PREFIX:
        if prov.startswith(k): return v
    return None
def base_of(rest):
    m=re.search(r'([가-힣]+시)',rest) or re.search(r'([가-힣]+군)',rest) or re.search(r'([가-힣]+구)',rest)
    return m.group(1) if m else None
def key_from_addr(addr):
    a=str(addr).replace('(산지)','').strip(); toks=a.split()
    if len(toks)<2: return None
    prov=toks[0]; rest=' '.join(toks[1:]); base=base_of(rest)
    if '광주통합' in prov or prov.startswith('전남광주'):
        sido='29' if (base and base.endswith('구')) else '46'
    else:
        sido=sido_of(prov)
    if sido is None: return None
    return sido if sido in METRO else (f"{sido}|{base}" if base else None)
def key_from_signgu(code,nm):
    sido=str(code).zfill(5)[:2]; base=str(nm).split()[0]
    return sido if sido in METRO else f"{sido}|{base}"

def main():
    # ---- 1. 지점(ASOS/AWS) → key ----
    meta=pd.read_csv(META,encoding='cp949',skiprows=1); meta.columns=[c.strip() for c in meta.columns]
    meta['종료일']=meta['종료일'].fillna('9999-12-31')
    meta=meta.sort_values(['지점','종료일']).groupby('지점',as_index=False).last().set_index('지점')
    def stn_key(s):
        a=meta.loc[s,'지점주소'] if s in meta.index else None
        return key_from_addr(a) if isinstance(a,str) else None

    # ---- 2. 기상 정규화 로드 + key ----
    met=[]
    for f in ['asos_2023_2025_normalized.csv','aws_2023_2025_normalized.csv']:
        d=pd.read_csv(BASE+f, usecols=['station','date','S_temp','S_humidity','S_wind','S_precip'])
        met.append(d)
    met=pd.concat(met,ignore_index=True)
    met['key']=met['station'].map(stn_key)
    met=met.dropna(subset=['key'])
    wx=met.groupby(['key','date'])[['S_temp','S_humidity','S_wind','S_precip']].mean().reset_index()
    print("기상 (key,date):",len(wx))

    # ---- 3. 대기질(AirKorea) → key ----
    aksrc=pd.read_excel(AK_KEYSRC,engine="calamine",usecols=["측정소코드","주소"]).dropna().drop_duplicates("측정소코드")
    aksrc['key']=aksrc['주소'].apply(key_from_addr)
    code2key=dict(zip(aksrc['측정소코드'],aksrc['key']))
    ak=pd.read_csv(BASE+'airkorea_2023_2025_normalized.csv',usecols=['station','date','S_airquality','S_aq_mean'])
    ak['key']=ak['station'].map(code2key); ak=ak.dropna(subset=['key'])
    aq=ak.groupby(['key','date'])[['S_airquality','S_aq_mean']].mean().reset_index()
    print("대기질 (key,date):",len(aq))

    # ---- 4. 관광 피벗 ----
    t=pd.read_csv(TOUR,encoding='utf-8-sig')
    t['date']=pd.to_datetime(t['baseYmd'],format='%Y%m%d').dt.strftime('%Y-%m-%d')
    piv=t.pivot_table(index=['signguCode','signguNm','date'],columns='touDivNm',values='touNum',aggfunc='sum').reset_index()
    piv.columns.name=None
    ren={'현지인(a)':'visitor_local','외지인(b)':'visitor_domestic','외국인(c)':'visitor_foreign'}
    piv=piv.rename(columns=ren)
    vcols=[c for c in ['visitor_local','visitor_domestic','visitor_foreign'] if c in piv.columns]
    piv['visitor_total']=piv[vcols].sum(axis=1)
    piv['key']=[key_from_signgu(c,n) for c,n in zip(piv['signguCode'],piv['signguNm'])]
    print("관광 (시군구,date):",len(piv))

    # ---- 5. 조인 ----
    m=piv.merge(wx,on=['key','date'],how='left').merge(aq,on=['key','date'],how='left')
    cols=['signguCode','signguNm','date','visitor_local','visitor_domestic','visitor_foreign','visitor_total',
          'S_temp','S_humidity','S_wind','S_precip','S_airquality','S_aq_mean']
    cols=[c for c in cols if c in m.columns]
    m=m[cols].sort_values(['signguCode','date'])
    m.to_csv('/home/claude/master_tourism_weather_2023_2025.csv',index=False,encoding='utf-8-sig')
    # 커버리지 리포트
    print("\n=== 마스터 ===")
    print("행:",len(m),"| 시군구:",m['signguCode'].nunique(),"| 기간:",m['date'].min(),"~",m['date'].max())
    for c in ['S_temp','S_humidity','S_wind','S_precip','S_airquality']:
        print(f"  {c:14s} 결측 {m[c].isna().mean()*100:4.1f}%")
    # 완전 결측 시군구
    wxmiss=m.groupby('signguNm')['S_temp'].apply(lambda s:s.isna().all())
    aqmiss=m.groupby('signguNm')['S_airquality'].apply(lambda s:s.isna().all())
    print("기상 전무 시군구:", list(wxmiss[wxmiss].index))
    print("대기질 전무 시군구 수:", int(aqmiss.sum()), "예:", list(aqmiss[aqmiss].index)[:15])

if __name__=="__main__": main()


## 3. 계절별 히든스팟 (`03_hidden_spot.py`)

안 붐비고(혼잡도↓) 날씨 좋고(KTCI ≥ 계절 중앙값) 볼거리 있는(매력도↑) 시군구.
`히든점수 = 매력도(z(다양성)+z(소비)+z(외지인)) − 혼잡도(z(log 방문객))`

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
03_hidden_spot.py
=================
계절별 '히든스팟' 도출.

정의 : 그 계절에 (1) 안 붐비고(혼잡도 낮음) (2) 날씨가 좋고(KTCI ≥ 그 계절 중앙값)
       (3) 볼거리가 있는(매력도 높음) 시군구.

점수 산출
    혼잡도  crowd   = z( log(방문객수) )                 # 낮을수록 여유
    매력도  appeal  = z(관광객다양성) + z(관광소비강도) + z(타권역방문자비중)   # 세 지표 동일 가중
    히든점수 hidden = appeal − crowd                     # 좋은 날씨 통과 지역 중 상위

핵심 : 단위가 다른 지표를 표준화(z)로 같은 잣대에 맞춘 뒤, 계절마다 따로 순위화한다.

입력  : master_monthly_ktci_tourism.csv   (날씨×관광 월별 마스터)
출력  : hidden_by_season.csv              (계절별 상위 후보)

의존성 : pandas, numpy
사용법 : python 03_hidden_spot.py --master ../dataset/master_monthly_ktci_tourism.csv
"""
from __future__ import annotations
import argparse
import numpy as np
import pandas as pd

SEASONS = {12: "겨울", 1: "겨울", 2: "겨울",
           3: "봄", 4: "봄", 5: "봄",
           6: "여름", 7: "여름", 8: "여름",
           9: "가을", 10: "가을", 11: "가을"}


def zscore(s: pd.Series) -> pd.Series:
    """표준화 (평균 0, 표준편차 1). 표준편차 0이면 0으로."""
    sd = s.std(ddof=0)
    return (s - s.mean()) / sd if sd and not np.isnan(sd) else s * 0.0


def build_hidden(df: pd.DataFrame, ktci_col: str = "KTCI_hybrid", top_n: int = 12) -> pd.DataFrame:
    """월별 마스터 → 계절별 히든스팟 순위표."""
    df = df.copy()
    df["month"] = df["baseYm"].astype(str).str[-2:].astype(int)
    df["season"] = df["month"].map(SEASONS)

    # 시군구×계절 평균 (여러 달을 계절로 집계)
    agg = (df.groupby(["season", "signguCode", "signguNm"])
             .agg(KTCI=(ktci_col, "mean"),
                  visitor=("visitor_total", "mean"),
                  diversity=("관광객다양성", "mean"),
                  spend=("관광소비강도", "mean"),
                  outside=("타권역방문자비중", "mean"))
             .reset_index())

    out = []
    for season, g in agg.groupby("season"):
        g = g.dropna(subset=["KTCI", "visitor", "diversity", "spend", "outside"]).copy()
        # (2) 좋은 날씨 필터 : 그 계절 KTCI 중앙값 이상
        g = g[g["KTCI"] >= g["KTCI"].median()]
        # (1) 안 붐빔 필터 : 방문객 중앙값 이하(비혼잡군)만 후보로
        g["crowd"] = zscore(np.log1p(g["visitor"]))
        g = g[g["visitor"] <= g["visitor"].median()]
        # (3) 매력도 : 비혼잡군 안에서 표준화해 합산, 매력에서 혼잡을 빼 최종 점수
        g["appeal"] = zscore(g["diversity"]) + zscore(g["spend"]) + zscore(g["outside"])
        g["hidden_score"] = g["appeal"] - zscore(np.log1p(g["visitor"]))
        g = g.sort_values("hidden_score", ascending=False).head(top_n)
        out.append(g)

    res = pd.concat(out, ignore_index=True)
    order = {"봄": 0, "여름": 1, "가을": 2, "겨울": 3}
    return res.sort_values(["season", "hidden_score"],
                           key=lambda c: c.map(order) if c.name == "season" else c,
                           ascending=[True, False]).reset_index(drop=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--master", default="../dataset/master_monthly_ktci_tourism.csv")
    ap.add_argument("--ktci", default="KTCI_hybrid",
                    help="사용할 KTCI 컬럼 (KTCI_data / KTCI_2014_adapted / KTCI_hybrid)")
    ap.add_argument("--top", type=int, default=12)
    ap.add_argument("--out", default="hidden_by_season.csv")
    args = ap.parse_args()

    df = pd.read_csv(args.master)
    res = build_hidden(df, ktci_col=args.ktci, top_n=args.top)
    res.to_csv(args.out, index=False, encoding="utf-8-sig")

    print(f"[OK] {args.out} 저장 ({len(res)}행)")
    for season in ["봄", "여름", "가을", "겨울"]:
        top = res[res.season == season].head(1)
        if len(top):
            r = top.iloc[0]
            print(f"  {season} 1위 : {r.signguNm}  (KTCI {r.KTCI:.1f}, 히든점수 {r.hidden_score:.2f})")


if __name__ == "__main__":
    main()


## 4. 데이터 기반 KTCI 산출·검증 (`KTCI_code/src/run_pipeline.py`) — 최유성

2023~2025 관광×기상 결합자료로 **계절별 데이터 기반 KTCI**를 산출하고, 2014 설문 가중치 adapted benchmark·민감도 모형과 비교·검증하는 재현 파이프라인. 관측률·Spearman·10분위 반응폭을 결합한 가중치, hybrid 결합(계절별 α).

> 전체 저장소(오케스트레이터 `run_all.py`, 민감도 비교 스크립트, 테스트)는 `KTCI_code/` 참고.

In [ ]:
from __future__ import annotations

import argparse
import hashlib
import json
import math
from pathlib import Path
import shutil
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

warnings.filterwarnings("ignore", category=FutureWarning)

SEASON_KO = {"spring": "봄", "summer": "여름", "autumn": "가을", "winter": "겨울"}
COMP_KO = {
    "thermal": "온열",
    "humidity": "습도",
    "wind": "바람",
    "precipitation": "강수",
    "air_quality": "대기질",
}


def load_config(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")


def weighted_available(scores: pd.DataFrame, weights: dict[str, float], min_n: int = 2) -> pd.Series:
    cols = [c for c in weights if c in scores.columns]
    w = pd.Series({c: weights[c] for c in cols}, dtype=float)
    valid = scores[cols].notna()
    numerator = scores[cols].mul(w, axis=1).sum(axis=1, min_count=1)
    denominator = valid.mul(w, axis=1).sum(axis=1)
    out = numerator.div(denominator).where(valid.sum(axis=1) >= min_n)
    return out


def top_bottom(y: pd.Series, index: pd.Series, q: float) -> tuple[float, float, float, int]:
    d = pd.DataFrame({"y": y, "index": index}).dropna()
    if len(d) < 30 or d["index"].nunique() < 5:
        return np.nan, np.nan, np.nan, len(d)
    lo, hi = d["index"].quantile([q, 1 - q])
    bottom = d.loc[d["index"] <= lo, "y"].mean()
    top = d.loc[d["index"] >= hi, "y"].mean()
    return top, bottom, top - bottom, len(d)


def performance(df: pd.DataFrame, index_col: str, q: float, train_years: list[int], test_years: list[int]) -> dict:
    d = df[["year", "tourism_residual_log", "tourism_change_pct", index_col]].dropna()
    rho = d[[index_col, "tourism_residual_log"]].corr(method="spearman").iloc[0, 1] if len(d) else np.nan
    top, bottom, diff, n = top_bottom(d["tourism_change_pct"], d[index_col], q)
    tr = d[d.year.isin(train_years)]
    te = d[d.year.isin(test_years)]
    if len(tr) >= 30 and len(te) >= 30:
        model = LinearRegression().fit(tr[[index_col]], tr["tourism_residual_log"])
        pred = model.predict(te[[index_col]])
        r2 = r2_score(te["tourism_residual_log"], pred)
        mae = mean_absolute_error(te["tourism_residual_log"], pred)
    else:
        r2 = mae = np.nan
    return {
        "index": index_col, "n": n, "spearman": rho,
        "top20_mean_pct": top, "bottom20_mean_pct": bottom,
        "top_bottom_diff_pp": diff, "test_r2": r2, "test_mae_log": mae,
    }


def region_group(code: int) -> str:
    p = int(code) // 1000
    first2 = int(code) // 1000
    if first2 in (11, 28, 41): return "수도권"
    if first2 in (42, 51): return "강원권"
    if first2 in (30, 36, 43, 44): return "충청권"
    if first2 in (29, 45, 46, 52): return "호남권"
    if first2 in (26, 27, 31, 47, 48): return "영남권"
    if first2 in (50,): return "제주권"
    return "기타"


def plot_save(fig, stem: Path, formats: list[str], dpi: int) -> None:
    stem.parent.mkdir(parents=True, exist_ok=True)
    for fmt in formats:
        fig.savefig(stem.with_suffix("." + fmt), dpi=dpi, bbox_inches="tight")
    plt.close(fig)


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    args = ap.parse_args()
    config_path = Path(args.config).resolve()
    root = config_path.parent
    cfg = load_config(config_path)
    out = root / cfg["project"]["output_dir"]
    tables, figs, logs, diagnostics = out / "tables", out / "figures", out / "logs", out / "diagnostics"
    for p in (tables, figs, logs, diagnostics):
        p.mkdir(parents=True, exist_ok=True)

    style = cfg["style"]
    plt.rcParams["font.family"] = style["font_family"]
    plt.rcParams["axes.unicode_minus"] = False
    sns.set_theme(style="whitegrid", font=style["font_family"])

    input_path = root / cfg["project"]["input_file"]
    if not input_path.exists():
        raise FileNotFoundError(f"입력 파일이 없습니다: {input_path}")
    raw_hash = hashlib.sha256(input_path.read_bytes()).hexdigest()
    df = pd.read_csv(input_path, encoding="utf-8-sig")
    required = ["signguCode", "signguNm", "date", "visitor_total", "S_temp", "S_humidity", "S_wind", "S_precip", "S_airquality"]
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError(f"필수 열 누락: {missing_cols}")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["weekday"] = df["date"].dt.weekday
    month_to_season = {m: s for s, months in cfg["analysis"]["seasons"].items() for m in months}
    df["season"] = df["month"].map(month_to_season)
    df["region_group"] = df["signguCode"].map(region_group)
    stress_map = cfg["analysis"]["stress_columns"]
    stress_cols = list(stress_map.values())
    for c in stress_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    quality = []
    for c in df.columns:
        x = df[c]
        quality.append({
            "variable": c, "dtype": str(x.dtype), "n": len(x), "missing_n": int(x.isna().sum()),
            "missing_pct": float(x.isna().mean() * 100), "unique_n": int(x.nunique(dropna=True)),
            "min": float(x.min()) if pd.api.types.is_numeric_dtype(x) and x.notna().any() else None,
            "max": float(x.max()) if pd.api.types.is_numeric_dtype(x) and x.notna().any() else None,
        })
    quality_df = pd.DataFrame(quality)
    save_csv(quality_df, tables / "step1_data_quality.csv")

    stress_range = pd.DataFrame([{
        "variable": c, "below_0_n": int((df[c] < 0).sum()), "above_1_n": int((df[c] > 1).sum()),
        "min": df[c].min(), "max": df[c].max()
    } for c in stress_cols])
    save_csv(stress_range, diagnostics / "stress_range_check.csv")

    haman = df[df["signguCode"] == 48730]
    haman_diag = pd.DataFrame([{
        "issue": "함안군 기상 결측", "signguCode": 48730, "rows": len(haman),
        **{f"{c}_missing_pct": float(haman[c].isna().mean() * 100) if len(haman) else np.nan for c in stress_cols},
        "action": "원자료를 임의 보간하지 않고 weighted_available로 관측 가능한 대기질만 사용. 최소 2개 영역 조건 때문에 KTCI는 결측 처리."
    }])
    save_csv(haman_diag, diagnostics / "haman_missing_diagnostic.csv")

    hierarchy = df[["signguCode", "signguNm"]].drop_duplicates().copy()
    hierarchy["is_city_level_name"] = hierarchy["signguNm"].astype(str).str.endswith("시")
    hierarchy["possible_overlap_group"] = hierarchy["signguCode"].astype(str).str[:3]
    counts = hierarchy.groupby("possible_overlap_group")["signguCode"].transform("count")
    hierarchy["possible_parent_child_overlap"] = hierarchy["is_city_level_name"] & (counts > 1)
    hierarchy["aggregation_policy"] = "관광객 합산 분석에서 제외; 행 단위 상관·잔차 분석만 사용"
    save_csv(hierarchy, diagnostics / "administrative_hierarchy_diagnostic.csv")

    df["log_visitors"] = np.log1p(df["visitor_total"].clip(lower=0))
    gcols = cfg["analysis"]["baseline_group"]
    grp = df.groupby(gcols, dropna=False)["log_visitors"]
    gsum, gcount = grp.transform("sum"), grp.transform("count")
    df["baseline_log_loo"] = (gsum - df["log_visitors"]) / (gcount - 1)
    df.loc[gcount < 2, "baseline_log_loo"] = np.nan
    df["tourism_residual_log"] = df["log_visitors"] - df["baseline_log_loo"]
    df["tourism_change_pct"] = np.expm1(df["tourism_residual_log"]) * 100

    corr = df[stress_cols].corr(method="spearman")
    corr.to_csv(tables / "step2_stress_correlation.csv", encoding="utf-8-sig")

    target_rows = []
    for comp, c in stress_map.items():
        sub = df[[c, "tourism_residual_log"]].dropna()
        rho = sub.corr(method="spearman").iloc[0, 1]
        low = sub[c].quantile(.2); high = sub[c].quantile(.8)
        lo_y = sub.loc[sub[c] <= low, "tourism_residual_log"].mean()
        hi_y = sub.loc[sub[c] >= high, "tourism_residual_log"].mean()
        target_rows.append({
            "component": comp, "variable": c, "n": len(sub), "spearman_stress_vs_tourism": rho,
            "low_stress_mean_log": lo_y, "high_stress_mean_log": hi_y,
            "high_minus_low_log": hi_y - lo_y, "availability": len(sub) / len(df)
        })
    target_corr = pd.DataFrame(target_rows)
    save_csv(target_corr, tables / "step3_stress_tourism_relationship.csv")

    weight_rows = []
    curve_rows = []
    for season, sdf in df.groupby("season"):
        ev = []
        for comp, c in stress_map.items():
            sub = sdf[[c, "tourism_residual_log"]].dropna()
            rho = sub.corr(method="spearman").iloc[0, 1] if len(sub) else np.nan
            q = sub.assign(bin=pd.qcut(sub[c], q=10, duplicates="drop")).groupby("bin", observed=True).agg(
                stress_mean=(c, "mean"), tourism_mean=("tourism_residual_log", "mean"), n=(c, "size")
            ).reset_index(drop=True) if len(sub) else pd.DataFrame()
            amplitude = float(q["tourism_mean"].max() - q["tourism_mean"].min()) if len(q) else np.nan
            availability = len(sub) / len(sdf) if len(sdf) else np.nan
            evidence = math.sqrt(max(abs(rho), 1e-8) * max(amplitude, 1e-8) * max(availability, 1e-8))
            direction_ok = bool(rho <= 0) if pd.notna(rho) else False
            ev.append((comp, evidence))
            weight_rows.append({
                "season": season, "component": comp, "variable": c, "n": len(sub), "spearman": rho,
                "decile_amplitude_log": amplitude, "availability": availability,
                "direction_expected_negative": direction_ok, "evidence": evidence,
            })
            if len(q):
                q["season"], q["component"] = season, comp
                curve_rows.append(q)
        total = sum(v for _, v in ev)
        for comp, evidence in ev:
            for row in reversed(weight_rows):
                if row["season"] == season and row["component"] == comp:
                    row["data_weight"] = evidence / total if total else np.nan
                    break
    weights = pd.DataFrame(weight_rows)
    save_csv(weights, tables / "step4_seasonal_weight_decomposition.csv")
    if curve_rows:
        save_csv(pd.concat(curve_rows, ignore_index=True), tables / "step6_decile_response_curves.csv")

    scores = pd.DataFrame(index=df.index)
    for comp, c in stress_map.items():
        scores[comp] = (1 - df[c]).clip(0, 1) * 100
        df[f"score_{comp}"] = scores[comp]

    for season in cfg["analysis"]["seasons"]:
        idx = df["season"] == season
        wdata = weights[weights.season.eq(season)].set_index("component")["data_weight"].to_dict()
        df.loc[idx, "KTCI_data"] = weighted_available(scores.loc[idx], wdata, cfg["missing"]["minimum_available_components"])
        raw2014 = cfg["benchmark_2014"][season].copy()
        raw2014.pop("cloud_unavailable", None)
        common = {"thermal": raw2014["thermal"], "precipitation": raw2014["precipitation"], "wind": raw2014["wind"]}
        s = sum(common.values()); w2014 = {k: v / s for k, v in common.items()}
        df.loc[idx, "KTCI_2014_adapted"] = weighted_available(scores.loc[idx], w2014, 2)

    alpha_rows = []
    chosen = {}
    for season in cfg["analysis"]["seasons"]:
        d = df[(df.season == season) & df.year.isin(cfg["analysis"]["train_years"])].copy()
        best_alpha, best_rho = 0.0, -np.inf
        for alpha in cfg["analysis"]["hybrid_alpha_grid"]:
            hybrid = alpha * d["KTCI_2014_adapted"] + (1 - alpha) * d["KTCI_data"]
            rho = pd.DataFrame({"h": hybrid, "y": d["tourism_residual_log"]}).corr(method="spearman").iloc[0, 1]
            alpha_rows.append({"season": season, "alpha_survey": alpha, "train_spearman": rho})
            if pd.notna(rho) and rho > best_rho:
                best_alpha, best_rho = alpha, rho
        chosen[season] = best_alpha
        idx = df.season.eq(season)
        df.loc[idx, "KTCI_hybrid"] = best_alpha * df.loc[idx, "KTCI_2014_adapted"] + (1 - best_alpha) * df.loc[idx, "KTCI_data"]
    alpha_df = pd.DataFrame(alpha_rows)
    alpha_df["selected"] = alpha_df.apply(lambda r: r.alpha_survey == chosen[r.season], axis=1)
    save_csv(alpha_df, tables / "step7_hybrid_alpha_search.csv")

    perf_rows = []
    for season in ["all"] + list(cfg["analysis"]["seasons"]):
        sdf = df if season == "all" else df[df.season.eq(season)]
        for region in ["전국"] + sorted(x for x in df.region_group.dropna().unique() if x != "기타"):
            rdf = sdf if region == "전국" else sdf[sdf.region_group.eq(region)]
            for model in ["KTCI_data", "KTCI_2014_adapted", "KTCI_hybrid"]:
                row = performance(rdf, model, cfg["analysis"]["top_bottom_quantile"], cfg["analysis"]["train_years"], cfg["analysis"]["test_years"])
                row.update({"season": season, "region_group": region})
                perf_rows.append(row)
    perf = pd.DataFrame(perf_rows)
    save_csv(perf, tables / "step7_index_performance.csv")

    year_perf = []
    for year, ydf in df.groupby("year"):
        for model in ["KTCI_data", "KTCI_2014_adapted", "KTCI_hybrid"]:
            row = performance(ydf, model, cfg["analysis"]["top_bottom_quantile"], [year], [year])
            row["year"] = year
            year_perf.append(row)
    save_csv(pd.DataFrame(year_perf), tables / "step7_year_stability.csv")

    scored_cols = ["signguCode", "signguNm", "date", "year", "season", "region_group", "visitor_total",
                   "tourism_residual_log", "tourism_change_pct"] + stress_cols + \
                  [f"score_{c}" for c in stress_map] + ["KTCI_data", "KTCI_2014_adapted", "KTCI_hybrid"]
    save_csv(df[scored_cols].head(50000), tables / "scored_observations_sample.csv")

    colors = style["colors"]; formats = style["formats"]; dpi = style["dpi"]
    qplot = quality_df[quality_df.variable.isin(stress_cols)].sort_values("missing_pct")
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.barplot(data=qplot, x="missing_pct", y="variable", color=colors["data_driven"], ax=ax)
    ax.set(title="스트레스 스코어 결측률", xlabel="결측률 (%)", ylabel="")
    plot_save(fig, figs / "step1_missing_rates", formats, dpi)

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
    ax.set_title("스트레스 스코어 간 Spearman 상관 관계")
    plot_save(fig, figs / "step2_stress_correlation", formats, dpi)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    t = target_corr.sort_values("spearman_stress_vs_tourism")
    sns.barplot(data=t, x="spearman_stress_vs_tourism", y="component", color=colors["data_driven"], ax=ax)
    ax.axvline(0, color="#333333", lw=1)
    ax.set(
        title="스트레스 스코어와 혼잡도 잔차의 상관 관계",
        xlabel="Spearman 상관 관계 (음수일수록 스트레스 스코어 증가 시 관광 감소)",
        ylabel="",
    )
    plot_save(fig, figs / "step3_stress_tourism", formats, dpi)

    wp = weights.pivot(index="component", columns="season", values="data_weight")
    wp = wp[[s for s in cfg["analysis"]["seasons"] if s in wp.columns]]
    fig, ax = plt.subplots(figsize=(10, 5))
    wp.plot(kind="bar", ax=ax, color=[colors[s] for s in wp.columns])
    ax.set(title="계절별 데이터 기반 가중치", xlabel="", ylabel="가중치")
    ax.legend([SEASON_KO.get(s, s) for s in wp.columns], title="계절")
    plt.xticks(rotation=0)
    plot_save(fig, figs / "step4_seasonal_weights", formats, dpi)

    regional = perf[(perf.season == "all")].pivot(index="region_group", columns="index", values="spearman")
    fig, ax = plt.subplots(figsize=(10, 5))
    regional.plot(kind="bar", ax=ax, color=[colors["benchmark_2014"], colors["data_driven"], colors["hybrid"]])
    ax.set(title="권역별 지수 Spearman 상관 관계 비교", xlabel="", ylabel="Spearman 상관 관계")
    plt.xticks(rotation=0)
    plot_save(fig, figs / "step5_regional_performance", formats, dpi)

    nat = perf[(perf.region_group == "전국") & (perf.season != "all")].copy()
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=nat, x="season", y="top_bottom_diff_pp", hue="index",
                palette=[colors["data_driven"], colors["benchmark_2014"], colors["hybrid"]], ax=ax)
    ax.set(title="계절별 상·하위 20% 관광 변화 차이", xlabel="", ylabel="차이 (%p)")
    plot_save(fig, figs / "step7_seasonal_top_bottom", formats, dpi)

    fig, ax = plt.subplots(figsize=(9, 5))
    for season, adf in alpha_df.groupby("season"):
        ax.plot(adf.alpha_survey, adf.train_spearman, marker="o", ms=3, label=SEASON_KO[season], color=colors[season])
    ax.set(
        title="Hybrid α 탐색 (2023–2024 학습)",
        xlabel="설문 benchmark 비중 α",
        ylabel="학습 Spearman 상관 관계",
    )
    ax.legend()
    plot_save(fig, figs / "step7_hybrid_alpha_search", formats, dpi)

    selected_alpha = ", ".join(f"{SEASON_KO[s]} {a:.2f}" for s, a in chosen.items())
    national = perf[(perf.region_group == "전국") & (perf.season == "all")][
        ["index", "spearman", "top_bottom_diff_pp", "test_r2", "test_mae_log"]
    ]
    report = f"""# 3개년 계절별 KTCI EDA 최종 보고서


- 분석기간: {df.date.min().date()} ~ {df.date.max().date()}
- 관측치: {len(df):,}행, 시군구 코드 {df.signguCode.nunique():,}개
- 스트레스 정의: 0=좋음, 1=나쁨. 최종 적합도 점수는 `100 × (1-스트레스)`입니다.
- Hybrid의 설문 benchmark 비중 α(2023–2024 학습에서 선택): {selected_alpha}


1. **습도 33.94% 결측**: 임의 보간하지 않았습니다. 지수 계산 시 관측 가능한 구성요소의 가중치 합으로 다시 나누는 `weighted_available` 방식을 사용했습니다.
2. **함안군**: 온열·습도·바람·강수가 전부 결측이고 대기질만 존재했습니다. 최소 2개 영역이 필요하므로 함안군 KTCI는 결측으로 남겼습니다. 근거 없는 최근접 관측소 대체는 하지 않았습니다.
3. **시/구 계층 중복**: 관광객을 권역별로 합산하지 않았습니다. 각 행의 '평소 대비 관광 변화'를 계산한 뒤 상관과 구분력을 평가하여 이중합산을 피했습니다.
4. **2014 benchmark**: 원 논문의 계절별 가중치는 정확히 기록했지만, 현재 파일에 최고·평균기온과 운량 원변수가 없습니다. 따라서 온도 두 항목은 `thermal`로 합치고, 운량은 제외한 뒤 사용 가능한 공통영역에서 재정규화했습니다. 이는 **2014 계절별 가중치 adapted benchmark**이며 원 KTCI 완전 재현이 아닙니다.


결측률, 범위, 중복, 날짜, 행정계층을 점검했습니다. 스트레스 값은 모두 0~1 범위였습니다. 습도 결측은 구조적(AWS 비관측) 결측이므로 0으로 채우지 않았습니다.

Spearman 상관으로 동일한 나쁜 날씨가 여러 영역에 중복 반영되는 정도를 확인했습니다. 이 표는 인과관계가 아니라 정보 중복 진단입니다.

시군구·연도·월·요일이 같은 날짜끼리 '자기 자신을 제외한 평소 관광량'을 만들었습니다. 현재 날짜를 기준 평균에 넣지 않아 차이가 인위적으로 작아지는 것을 막았습니다.

각 계절에서 스트레스와 관광잔차의 순위상관, 스트레스 10분위 관광반응 폭, 데이터 가용률을 결합해 계절별 증거점수와 가중치를 만들었습니다.

관광객 수를 지역 간 합산하지 않고 권역별 행만 분리해 동일한 성능지표를 계산했습니다. 따라서 시/구 계층 중복이 합계에 이중 반영되지 않습니다.

원 기상값이 아니라 이미 piecewise-linear로 변환된 스트레스가 입력이므로, 새로운 Spline 점수함수를 덧씌우지 않았습니다. 대신 스트레스 10분위별 관광반응을 사용해 비선형성과 임계구간을 검증했습니다.

- Data-driven: 계절별 데이터 증거 가중치
- 2014 adapted benchmark: 동일한 0~100 점수에 2014 설문 계절가중치의 공통영역 재정규화
- Hybrid: 두 지수를 α로 결합하며 α는 2023–2024 학습기간에서만 선택, 2025년에 검증


{national.to_markdown(index=False)}

R²=1이면 관광잔차를 완벽히 예측하고, 0이면 평균 예측과 비슷하며, 음수이면 평균보다 못합니다. KTCI는 관광객 수 전체 예측모델이 아니므로 R²가 낮을 수 있습니다. 이 연구에서는 좋은 날과 나쁜 날의 순서를 보는 Spearman과 상·하위 20% 차이를 함께 봅니다.


- 현재 입력은 영역 스트레스만 포함하므로 원 기상값별 임계점 재추정은 불가능합니다.
- 2014 KTCI의 운량 점수를 직접 계산할 수 없어 adapted benchmark를 사용했습니다.
- 공휴일·축제·가격·교통 등 비기상 요인은 포함되지 않았습니다.
- Hybrid α는 더 긴 외부기간에서 재검증할 필요가 있습니다.
"""
    (root / "report").mkdir(exist_ok=True)
    (root / "report" / "KTCI_3Year_Final_Report.md").write_text(report, encoding="utf-8")

    metadata = {
        "input_file": str(input_path), "input_sha256": raw_hash, "rows": len(df),
        "columns": list(df.columns), "date_min": str(df.date.min()), "date_max": str(df.date.max()),
        "selected_hybrid_alpha": chosen, "python": sys.version,
    }
    (logs / "run_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    (logs / "run_complete.txt").write_text("SUCCESS\n", encoding="utf-8")
    print(json.dumps({"status": "SUCCESS", "rows": len(df), "selected_alpha": chosen}, ensure_ascii=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## 5. 오늘의 날씨 점수 (`service/app/weather_today.py`) — 이은민

전국 264개 시군구의 **실시간 날씨 점수**를 요청마다 새로 계산(정적 파일 없음). 대용량 원본 CSV 대신 미리 뽑아둔 계수·보간테이블(`config/at_coefficients.json`, `config/aq_stress_tables.json`)을 사용. 위치 기반 실시간 점수·5등급 분류·기상위험 자동 하향·활동 추천.

> API 키는 노트북에선 `os.environ` 으로 대체했습니다(실서버는 환경변수 주입).

In [ ]:
"""
전국 264개 시군구의 실시간 날씨 점수를 요청마다 새로 계산한다 (정적 파일 없음).

refresh_weather_data.py(로컬 스크립트)와 같은 계산이지만, 대용량 원본 CSV
대신 미리 뽑아둔 계수/보간테이블(config/at_coefficients.json,
config/aq_stress_tables.json)을 쓴다 - 서버리스 함수에 89MB짜리 CSV를
넣을 수는 없어서다.
"""

from __future__ import annotations

import asyncio
import json
import os
import re
from datetime import datetime, timedelta
from pathlib import Path

import httpx
import numpy as np

from .providers import now_kst

CONFIG_DIR = Path(__file__).resolve().parent.parent / "config"
KMA_AUTH_KEY = os.environ.get("KMA_AUTH_KEY", "")        # 실제 키는 서버 환경변수로 주입
SEOUL_AQ_AUTH_KEY = os.environ.get("SEOUL_AQ_AUTH_KEY", "")  # 실제 키는 서버 환경변수로 주입

with (CONFIG_DIR / "at_coefficients.json").open() as f:
    _AT = json.load(f)
COEF_DAY = np.array(_AT["coef_day"])
COEF_NIGHT = np.array(_AT["coef_night"])

with (CONFIG_DIR / "aq_stress_tables.json").open() as f:
    _AQ = json.load(f)
PM25_X, PM25_Y = np.array(_AQ["pm25_x"]), np.array(_AQ["pm25_y"])
PMC_X, PMC_Y = np.array(_AQ["pmc_x"]), np.array(_AQ["pmc_y"])

with (CONFIG_DIR / "signgu_centroids.json").open(encoding="utf-8") as f:
    REGIONS = json.load(f)

with (CONFIG_DIR / "seasonal_weights.json").open() as f:
    SEASON_WEIGHTS = json.load(f)

SEASONS = {"spring": [3, 4, 5], "summer": [6, 7, 8], "autumn": [9, 10, 11], "winter": [12, 1, 2]}
MONTH_TO_SEASON = {m: s for s, ms in SEASONS.items() for m in ms}

HEAT_ANCHORS = [(26, 0.0), (32, 0.25), (38, 0.50), (46, 0.75), (54, 1.0)]
COLD_ANCHORS_NEG = [(-9, 0.0), (0, 0.25), (13, 0.50), (27, 0.75), (40, 1.0)]
WIND_ANCHORS = [(3.3, 0.0), (8.0, 0.25), (13.9, 0.50), (17.2, 0.70), (21, 0.85), (24.5, 1.0)]
AMT_ANCHORS = [(0, 0.0), (10, 0.20), (30, 0.40), (50, 0.60), (80, 0.80), (150, 1.0)]
INT_ANCHORS = [(0, 0.0), (3, 0.25), (15, 0.50), (30, 0.75), (50, 1.0)]


def piecewise(x, anchors):
    xs = np.array([a[0] for a in anchors]); ys = np.array([a[1] for a in anchors])
    order = np.argsort(xs); xs, ys = xs[order], ys[order]
    return np.clip(np.interp(x, xs, ys, left=ys[0], right=ys[-1]), 0, 1)


def at_day_fn(ta, rh):
    ta = np.asarray(ta, dtype=float); rh = np.asarray(rh, dtype=float)
    out = ta.copy()
    mask = ta >= 27
    tm, rm = ta[mask], rh[mask]
    Xm = np.column_stack([np.ones_like(tm), tm, rm, tm*rm, tm**2, rm**2, tm**2*rm, tm*rm**2, tm**3, rm**3])
    out[mask] = Xm @ COEF_DAY
    return out


def at_night_fn(ta, ws):
    ta = np.asarray(ta, dtype=float); ws = np.asarray(ws, dtype=float)
    out = ta.copy()
    mask = ta < 11
    tm, wm = ta[mask], ws[mask]
    Xm = np.column_stack([np.ones_like(tm), tm, wm, tm*wm, tm**2, wm**2, tm**2*wm, tm*wm**2, tm**3])
    out[mask] = Xm @ COEF_NIGHT
    return out


def s_thermal(t1h, reh, wsd):
    heat = piecewise(at_day_fn(t1h, reh), HEAT_ANCHORS)
    cold = piecewise(-at_night_fn(t1h, wsd), COLD_ANCHORS_NEG)
    return np.maximum(heat, cold)


def thi_calc(ta, rh):
    return 1.8 * ta - 0.55 * (1 - rh / 100) * (1.8 * ta - 26) + 32


def s_humidity(t1h, reh):
    thi_s = piecewise(thi_calc(t1h, reh), [(68, 0.0), (75, 0.5), (80, 1.0)])
    rh_dev = np.clip(np.where(reh <= 40, (40 - reh) / 40, np.where(reh >= 60, (reh - 60) / 40, 0)), 0, 1)
    return np.maximum(thi_s, rh_dev)


def s_wind(wsd):
    return piecewise(wsd, WIND_ANCHORS)


def s_precip(rn1):
    return np.maximum(piecewise(rn1, AMT_ANCHORS), piecewise(rn1, INT_ANCHORS))


def s_pm25_fn(pm25):
    return np.clip(np.interp(pm25, PM25_X, PM25_Y, left=PM25_Y[0], right=PM25_Y[-1]), 0, 1)


def s_pmc_fn(pmc):
    return np.clip(np.interp(pmc, PMC_X, PMC_Y, left=PMC_Y[0], right=PMC_Y[-1]), 0, 1)


def s_airquality(pm25, pm10):
    pmc = max(pm10 - pm25, 0.0)
    return max(float(s_pm25_fn(pm25)), float(s_pmc_fn(pmc)))


GRADE_HEAD = {
    "아주좋음": "실외 활동하기 아주 좋은 날씨예요", "좋음": "실외 활동에 좋은 날씨예요",
    "무난": "실외 활동하기 무난해요", "주의": "가벼운 실외 활동은 괜찮아요", "나쁨": "실외 활동은 조금 주의하세요",
}


def grade_of_score(k):
    if k is None or np.isnan(k):
        return ""
    return "아주좋음" if k >= 85 else "좋음" if k >= 70 else "무난" if k >= 60 else "주의" if k >= 50 else "나쁨"


def danger_factors(at_day, at_night, pm25, pm10, rn_day, rn_hr1, ws_max):
    d = []
    if at_day is not None and at_day >= 35:
        d.append((3, "폭염, 온열질환 주의"))
    if at_night is not None and at_night <= -12:
        d.append((3, "한파, 한랭질환 주의"))
    if (pm25 is not None and pm25 >= 76) or (pm10 is not None and pm10 >= 151):
        d.append((3, "미세먼지 매우 나쁨, 마스크 착용"))
    if (rn_day is not None and rn_day >= 80) or (rn_hr1 is not None and rn_hr1 >= 30):
        d.append((4, "호우, 침수·안전 주의"))
    if ws_max is not None and ws_max >= 21:
        d.append((3, "강풍, 안전 주의"))
    return d


def gentle_notes(at_day, at_night, pm25, pm10, rn_day, ws_max):
    n = []
    if at_day is not None and 33 <= at_day < 35:
        n.append("더위 주의")
    elif at_night is not None and -12 < at_night <= -6:
        n.append("쌀쌀해요, 옷 챙기세요")
    if (pm25 is not None and 36 <= pm25 < 76) or (pm10 is not None and 81 <= pm10 < 151):
        n.append("미세먼지 주의")
    if rn_day is not None and 30 <= rn_day < 80:
        n.append("비 많음, 우산 필수")
    elif rn_day is not None and 5 <= rn_day < 30:
        n.append("비 소식, 우산 챙기세요")
    if ws_max is not None and 14 <= ws_max < 21:
        n.append("바람 강함")
    return n


def build_display(ktci, at_day, at_night, pm25, pm10, rn_day, rn_hr1, ws_max):
    d = danger_factors(at_day, at_night, pm25, pm10, rn_day, rn_hr1, ws_max)
    if d:
        d.sort(key=lambda x: -x[0])
        phrases = [p for _, p in d]
        cap = 45 if len(d) >= 2 else 55
        grade = "나쁨" if len(d) >= 2 else "주의"
        return round(min(ktci, cap), 1), grade, f"⚠️ {', '.join(phrases)} — 실외 활동에 주의하세요"

    g = grade_of_score(ktci)
    notes = gentle_notes(at_day, at_night, pm25, pm10, rn_day, ws_max)
    if g in ("아주좋음", "좋음"):
        comment = f"{GRADE_HEAD[g]} · {notes[0]}" if notes else GRADE_HEAD[g]
    else:
        comment = f"{GRADE_HEAD[g]} — {', '.join(notes)}" if notes else GRADE_HEAD[g]
    return round(ktci, 1), g, comment


def season_weights_no_air(season):
    w = {k: v for k, v in SEASON_WEIGHTS[season].items() if k != "air_quality"}
    s = sum(w.values())
    return {k: v / s for k, v in w.items()}


def parse_rn1(val):
    if val in ("강수없음", "", None):
        return 0.0
    try:
        return float(val)
    except (TypeError, ValueError):
        m = re.search(r"([\d.]+)", str(val))
        return float(m.group(1)) if m else 0.0


def pick_base_time():
    now = now_kst()
    if now.minute < 40:
        now -= timedelta(hours=1)
    return now.strftime("%Y%m%d"), now.strftime("%H00")


async def fetch_ncst(client: httpx.AsyncClient, sem: asyncio.Semaphore, nx: int, ny: int, base_date: str, base_time: str):
    url = "https://apihub.kma.go.kr/api/typ02/openApi/VilageFcstInfoService_2.0/getUltraSrtNcst"
    params = {"pageNo": 1, "numOfRows": 20, "dataType": "JSON", "base_date": base_date,
              "base_time": base_time, "nx": nx, "ny": ny, "authKey": KMA_AUTH_KEY}
    async with sem:
        try:
            r = await client.get(url, params=params, timeout=10)
            data = r.json()
            body = data.get("response", {}).get("body")
            if not body:
                return (nx, ny), None
            items = {i["category"]: i["obsrValue"] for i in body["items"]["item"]}
            return (nx, ny), items
        except Exception:
            return (nx, ny), None


async def fetch_seoul_air_quality(client: httpx.AsyncClient):
    if not SEOUL_AQ_AUTH_KEY:
        return {}
    url = f"http://openAPI.seoul.go.kr:8088/{SEOUL_AQ_AUTH_KEY}/json/RealtimeCityAir/1/25/"
    try:
        r = await client.get(url, timeout=10)
        rows = r.json()["RealtimeCityAir"]["row"]
        out = {}
        for row in rows:
            name = row.get("MSRSTN_NM")
            pm10, pm25 = row.get("PM"), row.get("FPM")
            if name and pm10 not in (None, "", "-") and pm25 not in (None, "", "-"):
                out[name] = {"PM10": float(pm10), "PM25": float(pm25)}
        return out
    except Exception:
        return {}


_CACHE: dict = {"key": None, "data": None}
_LOCK = asyncio.Lock()
MIN_HEALTHY_DISTRICTS = 100  # 이보다 적게 받아오면 KMA 쪽 문제로 보고 이전 캐시를 유지한다

# 공유 캐시: 서버리스 인스턴스가 여러 개 떠도(동시접속 몰릴 때) 전부 같은 캐시를 보게 하기
# 위한 저장소. 인메모리 캐시(_CACHE)는 같은 인스턴스가 재사용될 때만 도움이 되고, 인스턴스가
# 여러 개 뜨면 각자 따로 KMA를 불러버린다 - 그걸 막아준다.
# 실제 데이터(60KB+)는 Vercel Blob에 "그 시간대 전용 경로"로 저장한다(덮어쓰기 하면 CDN이
# 예전 캐시를 한동안 계속 돌려주는 걸 확인해서, 시간대마다 새 경로를 씀). Edge Config에는
# 그 Blob의 URL만 담은 작은 "포인터"만 저장한다(Edge Config는 Hobby 플랜에서 전체 8KB
# 제한이라 264개 지역 데이터를 통째로 넣을 수 없다).
EDGE_CONFIG_ID = os.environ.get("EDGE_CONFIG_ID")
EDGE_CONFIG_READ_TOKEN = os.environ.get("EDGE_CONFIG_READ_TOKEN")
VERCEL_API_TOKEN = os.environ.get("VERCEL_API_TOKEN")
BLOB_READ_WRITE_TOKEN = os.environ.get("BLOB_READ_WRITE_TOKEN")
EDGE_CONFIG_ENABLED = bool(EDGE_CONFIG_ID and EDGE_CONFIG_READ_TOKEN and VERCEL_API_TOKEN)
BLOB_ENABLED = bool(BLOB_READ_WRITE_TOKEN)
SHARED_CACHE_ENABLED = EDGE_CONFIG_ENABLED and BLOB_ENABLED


async def _pointer_read(client: httpx.AsyncClient) -> dict | None:
    try:
        r = await client.get(
            f"https://edge-config.vercel.com/{EDGE_CONFIG_ID}/item/weather_pointer",
            params={"token": EDGE_CONFIG_READ_TOKEN}, timeout=5,
        )
        return r.json() if r.status_code == 200 else None
    except Exception:
        return None


async def _pointer_write(client: httpx.AsyncClient, base_date: str, base_time: str, blob_url: str) -> None:
    try:
        await client.patch(
            f"https://api.vercel.com/v1/edge-config/{EDGE_CONFIG_ID}/items",
            json={"items": [{"operation": "upsert", "key": "weather_pointer",
                             "value": {"baseDate": base_date, "baseTime": base_time, "blobUrl": blob_url}}]},
            headers={"Authorization": f"Bearer {VERCEL_API_TOKEN}"}, timeout=10,
        )
    except Exception:
        pass  # 포인터 쓰기 실패는 무시한다 - 다음 요청이 다시 계산하면 그만이다


async def _blob_write(client: httpx.AsyncClient, base_date: str, base_time: str, data: dict) -> str | None:
    pathname = f"weather_{base_date}_{base_time}.json"
    try:
        r = await client.put(
            f"https://blob.vercel-storage.com/{pathname}",
            content=json.dumps(data, ensure_ascii=False).encode("utf-8"),
            headers={"Authorization": f"Bearer {BLOB_READ_WRITE_TOKEN}", "x-api-version": "7",
                     "x-add-random-suffix": "0", "content-type": "application/json"},
            timeout=15,
        )
        return r.json().get("url") if r.status_code == 200 else None
    except Exception:
        return None


async def _blob_read(client: httpx.AsyncClient, url: str) -> dict | None:
    try:
        r = await client.get(url, timeout=8)
        return r.json() if r.status_code == 200 else None
    except Exception:
        return None


async def compute_all() -> dict:
    """KMA는 매시 40분에만 새 값을 발표하므로, 그 발표 주기(base_date/base_time)가 바뀔 때만
    다시 불러온다. 인메모리 캐시로 같은 인스턴스 재요청을 빠르게 막고, Blob+Edge Config
    포인터로 인스턴스가 여러 개 떠도(동시접속) 전부 같은 캐시를 공유하게 한다."""
    base_date, base_time = pick_base_time()
    key = (base_date, base_time)

    if _CACHE["key"] == key and _CACHE["data"] is not None:
        return _CACHE["data"]

    async with _LOCK:
        if _CACHE["key"] == key and _CACHE["data"] is not None:
            return _CACHE["data"]

        async with httpx.AsyncClient() as client:
            pointer = await _pointer_read(client) if SHARED_CACHE_ENABLED else None
            if pointer and (pointer.get("baseDate"), pointer.get("baseTime")) == key:
                shared = await _blob_read(client, pointer["blobUrl"])
                if shared:
                    _CACHE["key"] = key
                    _CACHE["data"] = shared
                    return shared

            fresh = await _compute_fresh(base_date, base_time)
            healthy = len(fresh["districts"]) >= MIN_HEALTHY_DISTRICTS
            # 성공 여부와 무관하게 이번 시간대는 "시도했음"으로 표시해 재요청마다 KMA를 다시
            # 부르지 않게 한다(한도 보호). 다만 결과가 부실하면 예전 공유 캐시가 있으면 그걸,
            # 없으면 인메모리 캐시라도 계속 내보낸다 - 텅 빈 화면보다 낫다.
            _CACHE["key"] = key
            if healthy:
                _CACHE["data"] = fresh
                if SHARED_CACHE_ENABLED:
                    blob_url = await _blob_write(client, base_date, base_time, fresh)
                    if blob_url:
                        await _pointer_write(client, base_date, base_time, blob_url)
            elif pointer:
                shared = await _blob_read(client, pointer["blobUrl"])
                if shared:
                    _CACHE["data"] = shared
                elif _CACHE["data"] is None:
                    _CACHE["data"] = fresh
            elif _CACHE["data"] is None:
                _CACHE["data"] = fresh
            return _CACHE["data"]


async def _compute_fresh(base_date: str, base_time: str) -> dict:
    season = MONTH_TO_SEASON[now_kst().month]
    weights = season_weights_no_air(season)
    weights_full = {k: v for k, v in SEASON_WEIGHTS[season].items()}

    unique_cells = sorted({(r["nx"], r["ny"]) for r in REGIONS})

    limits = httpx.Limits(max_connections=100, max_keepalive_connections=100)
    async with httpx.AsyncClient(limits=limits) as client:
        sem = asyncio.Semaphore(80)
        cell_results, seoul_aq = await asyncio.gather(
            asyncio.gather(*(fetch_ncst(client, sem, nx, ny, base_date, base_time) for nx, ny in unique_cells)),
            fetch_seoul_air_quality(client),
        )
    cell_obs = dict(cell_results)

    results = []
    for r in REGIONS:
        obs = cell_obs.get((r["nx"], r["ny"]))
        if obs is None:
            continue
        try:
            t1h = float(obs.get("T1H", "nan"))
            reh = float(obs.get("REH", "nan"))
            wsd = float(obs.get("WSD", "nan"))
        except ValueError:
            continue
        rn1 = parse_rn1(obs.get("RN1"))

        scores = {
            "thermal": (1 - s_thermal(np.array([t1h]), np.array([reh]), np.array([wsd]))[0]) * 100,
            "humidity": (1 - s_humidity(np.array([t1h]), np.array([reh]))[0]) * 100,
            "wind": (1 - s_wind(np.array([wsd]))[0]) * 100,
            "precipitation": (1 - s_precip(np.array([rn1]))[0]) * 100,
        }

        aq = seoul_aq.get(r["signguNm"]) if r["province"] == "서울특별시" else None
        if aq is not None:
            aq_stress = s_airquality(aq["PM25"], aq["PM10"])
            scores["air_quality"] = (1 - aq_stress) * 100
            ktci = sum(scores[k] * weights_full[k] for k in weights_full)
        else:
            ktci = sum(scores[k] * weights[k] for k in weights)

        at_day = float(at_day_fn(np.array([t1h]), np.array([reh]))[0])
        at_night = float(at_night_fn(np.array([t1h]), np.array([wsd]))[0])
        pm25 = aq["PM25"] if aq is not None else None
        pm10 = aq["PM10"] if aq is not None else None
        display_score, grade, comment = build_display(
            ktci=float(ktci), at_day=at_day, at_night=at_night,
            pm25=pm25, pm10=pm10, rn_day=rn1, rn_hr1=rn1, ws_max=wsd,
        )

        result = {
            "signguCode": r["signguCode"], "signguNm": r["signguNm"], "province": r["province"],
            "lat": r["lat"], "lon": r["lon"],
            "T1H": t1h, "REH": reh, "WSD": wsd, "RN1": rn1,
            "score": round(float(ktci), 1),
            "displayScore": display_score, "grade": grade, "comment": comment,
        }
        if aq is not None:
            result["PM10"] = aq["PM10"]
            result["PM25"] = aq["PM25"]
        results.append(result)

    return {
        "updatedAt": now_kst().strftime("%Y-%m-%d %H:%M"),
        "baseDate": base_date, "baseTime": base_time, "season": season,
        "districts": results,
    }


## 6. 여행 일정 동선 점수 (`service/app/trip.py`) — 신동준

'며칟날 어느 시군구를 순서대로 도는가'를 입력받아 **예보 기반 일자별 동선**을 채점. 지역-일 점수=그날 그 시군구 KTCI, 일자 점수=방문 시군구 평균, 여행 점수=일자 평균(날짜 동일 가중). 각 지역의 예보 범위 내 '더 좋은 날짜'도 함께 제시.

> 공유 코어(`scoring.py`·`subindex.py`·`aggregate.py`)와 FastAPI 라우팅(`main.py`)은 `service/app/` 참고.

In [ ]:
"""
일별 · 시군구별 여행 날씨 점수.

입력은 '며칟날 어느 시군구를 순서대로 도는가'다.

    지역-일 점수 = 그 시군구 좌표의 그날 KTCI
    일자 점수    = 그날 방문한 시군구 점수의 평균
    여행 점수    = 일자 점수의 평균 (날짜마다 같은 무게)

여행 점수를 지역-일 전체 평균이 아니라 일자 평균으로 잡은 이유: 하루에 여러 곳을 도는
날이 있으면 그날 날씨가 과대 반영되기 때문이다. 두 값 모두 응답에 담아 비교할 수 있게 했다.

계획 수정을 돕기 위해 각 지역마다 예보 범위 안의 '더 좋은 날짜'를 함께 계산한다.
어차피 예보를 통째로 받아 두므로 추가 호출 없이 나온다.
"""

from __future__ import annotations

from datetime import date

from .aggregate import build_daily, daily_coverage
from .providers import now_kst
from .response import _r
from .scoring import grade_of, score_day, season_of


def score_region_day(hourly: dict, day: date, variant: str = "hybrid") -> dict | None:
    """(시군구, 날짜) 하나. 예보가 없거나 영역이 부족하면 None."""
    daily = build_daily(hourly, day)
    res = score_day(daily, day, variant)
    if res.ktci is None:
        return None
    return {"result": res, "daily": daily}


def forecast_dates(hourly: dict) -> list[date]:
    return sorted({t.date() for t in hourly})


def alternatives(hourly: dict, current: date, variant: str, current_score: float,
                 limit: int = 3) -> list[dict]:
    """예보 범위 안에서 현재 날짜보다 점수가 높은 날들. 좋은 순."""
    out = []
    for d in forecast_dates(hourly):
        if d == current or d < now_kst().date():
            continue
        scored = score_region_day(hourly, d, variant)
        if scored is None:
            continue
        s = scored["result"].score_100
        if s > current_score + 1:          # 1점 이내 차이는 권할 이유가 없다
            out.append({"date": d.isoformat(), "score": s,
                        "gain": round(s - current_score, 1),
                        "grade": grade_of(scored["result"].ktci)})
    out.sort(key=lambda x: -x["score"])
    return out[:limit]


def build_trip(plan: list[dict], hourly_by_region: dict[str, dict],
               region_meta: dict[str, dict], variant: str = "hybrid",
               with_alternatives: bool = True) -> dict:
    """
    plan: [{"date": date, "regions": ["11110", ...]}, ...]   (순서 유지)
    hourly_by_region: {code: 시간별 예보}
    region_meta: {code: regions.get(code)}
    """
    days_out, day_scores, all_region_scores = [], [], []

    for entry in plan:
        day: date = entry["date"]
        regions_out = []

        for order, code in enumerate(entry["regions"], start=1):
            meta = region_meta[code]
            hourly = hourly_by_region.get(code, {})
            scored = score_region_day(hourly, day, variant)

            if scored is None:
                regions_out.append({
                    "order": order, "code": code, "name": meta["name"],
                    "label": meta["label"], "region_group": meta["region_group"],
                    "score": None, "grade": None,
                    "note": "예보 범위를 벗어났거나 관측 영역이 2개 미만이라 점수를 낼 수 없습니다.",
                    "alternatives": [],
                })
                continue

            res, daily = scored["result"], scored["daily"]
            score = res.score_100
            all_region_scores.append(score)
            top = res.top_component

            regions_out.append({
                "order": order, "code": code, "name": meta["name"],
                "label": meta["label"], "region_group": meta["region_group"],
                "score": score, "grade": grade_of(res.ktci), "ktci": _r(res.ktci),
                "subindices": _r(res.subindices), "weights_used": _r(res.weights_used),
                "missing": res.missing,
                "top_component": {"component": top[0], "share": _r(top[1])} if top else None,
                "daily_values": _r(dict(daily.__dict__)),
                "valid_hours": daily_coverage(hourly, day),
                "alternatives": alternatives(hourly, day, variant, score) if with_alternatives else [],
                "note": res.note,
            })

        scored_today = [r["score"] for r in regions_out if r["score"] is not None]
        day_score = round(sum(scored_today) / len(scored_today), 1) if scored_today else None
        if day_score is not None:
            day_scores.append(day_score)

        days_out.append({
            "date": day.isoformat(),
            "season": season_of(day.month),
            "score": day_score,
            "grade": grade_of(1 - day_score / 100) if day_score is not None else None,
            "regions": regions_out,
        })

    trip = round(sum(day_scores) / len(day_scores), 1) if day_scores else None
    flat = round(sum(all_region_scores) / len(all_region_scores), 1) if all_region_scores else None

    ranked = [(d["date"], r) for d in days_out for r in d["regions"] if r["score"] is not None]
    ranked.sort(key=lambda p: p[1]["score"])

    return {
        "summary": {
            "trip_score": trip,
            "grade": grade_of(1 - trip / 100) if trip is not None else None,
            "region_day_mean": flat,
            "day_count": len(days_out),
            "region_day_count": len(all_region_scores),
            "worst": _ref(ranked[0]) if ranked else None,
            "best": _ref(ranked[-1]) if ranked else None,
            "fixable": [
                {**_ref(p), "best_alternative": p[1]["alternatives"][0]}
                for p in ranked if p[1]["alternatives"]
            ][:5],
        },
        "variant": variant,
        "days": days_out,
        "method": ("지역-일 점수 = 그 시군구의 그날 KTCI · "
                   "일자 점수 = 그날 시군구 점수의 평균 · "
                   "여행 점수 = 일자 점수의 평균(날짜마다 같은 무게)"),
    }


def _ref(pair) -> dict:
    d, r = pair
    return {"date": d, "code": r["code"], "label": r["label"], "score": r["score"]}
